# 02 · Cleaning & Preprocessing
**Goal:** turn both raw files into clean, **hourly, UTC-indexed** tables with solar geometry and a capacity-normalised target. Every rule is logged.

Output: `data/processed/primary_hourly_clean.csv` and `data/processed/site_a_hourly_clean.csv`. These are plain CSV files that open in Excel.

In [1]:
import sys, warnings
sys.path.insert(0, "../src")          # project modules
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

from io_utils import PROCESSED, load_primary_raw, load_site_a_raw
from config import SITES, MIN_COVERAGE, SHORT_GAP_H, FLATLINE_H
from solar_geometry import add_solar_features

LOG = []   # (site, rule, rows affected)
def log(site, rule, n):
    LOG.append((site, rule, int(n)))

## Part A: Primary site
### A1. Load, drop unusable columns, rename
- `Wind_Speed` is 100 % empty.
- `Active_Power_Clean` contains imputed values (see notebook 01).
- Cumulative energy and current carry the same information as power.

In [2]:
cfg = SITES["primary"]
df = load_primary_raw().drop(columns=["Wind_Speed", "Active_Power_Clean",
                                      "Active_Energy_Delivered_Received", "Current_Phase_Average"])
df = df.rename(columns={
    "Active_Power": "power_kw", "Weather_Temperature_Celsius": "temp_c",
    "Weather_Relative_Humidity": "humidity_pct", "Global_Horizontal_Radiation": "ghi",
    "Diffuse_Horizontal_Radiation": "dhi", "Wind_Direction": "wind_dir",
    "Weather_Daily_Rainfall": "rain_daily_mm", "Radiation_Global_Tilted": "gti",
    "Radiation_Diffuse_Tilted": "dti"})
df.head()

,power_kw,temp_c,humidity_pct,ghi,dhi,wind_dir,rain_daily_mm,gti,dti
timestamp,,,,,,,,,
2024-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-01-01 00:05:00,0.0100,20.740343,18.584463,1.716115,1.296987,44.296715,0.0,NaN,NaN
2024-01-01 00:10:00,NaN,20.562437,18.965721,1.737423,1.339449,45.068665,0.0,NaN,NaN
2024-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.830827,0.929267
2024-01-01 00:20:00,0.0075,20.331274,18.677074,1.791579,1.389650,44.926125,0.0,3.675133,0.812759


### A2. Local time → UTC
The site has no daylight-saving time, so every local timestamp maps to exactly one UTC time.

In [3]:
df.index = df.index.tz_localize(cfg["tz"]).tz_convert("UTC")
log("primary", "duplicate timestamps dropped", df.index.duplicated().sum())
df = df[~df.index.duplicated()]
df.index[:3]

DatetimeIndex(['2023-12-31 14:30:00+00:00', '2023-12-31 14:35:00+00:00', '2023-12-31 14:40:00+00:00'], dtype='datetime64[ns, UTC]', name='timestamp', freq=None)

### A3. Physical bounds
Values outside physically possible limits are clipped (or set to NaN for wind direction).

In [4]:
bounds = {"humidity_pct": (0, 100), "ghi": (0, 1400), "dhi": (0, 800),
          "gti": (0, 1400), "dti": (0, 800), "temp_c": (-10, 50)}
for col, (lo, hi) in bounds.items():
    bad = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
    log("primary", f"{col} outside [{lo}, {hi}] clipped", bad.sum())
    df[col] = df[col].clip(lo, hi)

bad = df["wind_dir"].notna() & ((df["wind_dir"] < 0) | (df["wind_dir"] > 360))
log("primary", "wind_dir outside [0, 360] -> NaN", bad.sum())
df.loc[bad, "wind_dir"] = np.nan
pd.DataFrame(LOG, columns=["site", "rule", "rows"])

,site,rule,rows
0,primary,duplicate timestamps dropped,0
1,primary,"humidity_pct outside [0, 100] clipped",3783
2,primary,"ghi outside [0, 1400] clipped",7
3,primary,"dhi outside [0, 800] clipped",0
4,primary,"gti outside [0, 1400] clipped",0
5,primary,"dti outside [0, 800] clipped",0
6,primary,"temp_c outside [-10, 50] clipped",0
7,primary,"wind_dir outside [0, 360] -> NaN",1014


### A4. Rainfall: daily running total → rain per interval
`Weather_Daily_Rainfall` adds up during the day and resets at local midnight. Taking the difference gives the rain that fell in each interval.

In [5]:
local_day = df.index.tz_convert(cfg["tz"]).date
df["rain_mm"] = (df["rain_daily_mm"].groupby(local_day).diff().clip(lower=0)
                 .fillna(0).where(df["rain_daily_mm"].notna()))
df = df.drop(columns="rain_daily_mm")
print("Total rain (mm):", round(df["rain_mm"].sum(), 1))

Total rain (mm): 356.8


### A5. Resample 5-min → hourly
- Use the **mean** for power, irradiance and weather, the **sum** for rain, and a **circular mean** for wind direction (the average of 350° and 10° is 0°, not 180°).
- An hour is valid only if at least **80 %** of its 5-min samples exist. Otherwise it would be the average of a few samples and not representative.

In [6]:
grid = pd.date_range(df.index.min().floor("h"), df.index.max().ceil("h"), freq="5min", tz="UTC")
df5 = df.reindex(grid)                                    # makes gaps explicit

agg = {c: "mean" for c in df5.columns}; agg["rain_mm"] = "sum"
hourly = df5.resample("h").agg(agg)
rad = np.deg2rad(df5["wind_dir"])
hourly["wind_dir"] = np.rad2deg(np.arctan2(np.sin(rad).resample("h").mean(),
                                           np.cos(rad).resample("h").mean())) % 360

coverage = df5.notna().resample("h").mean()
for c in hourly.columns:
    weak = coverage[c] < MIN_COVERAGE
    log("primary", f"hourly {c}: <80% coverage -> NaN", (weak & hourly[c].notna()).sum())
    hourly.loc[weak, c] = np.nan
print(hourly.shape)
hourly.head()

(18746, 9)


,power_kw,temp_c,humidity_pct,ghi,dhi,wind_dir,gti,dti,rain_mm
2023-12-31 14:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2023-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Shared finishing steps (both sites)
1. **Solar geometry with pvlib:** solar elevation/zenith/azimuth, clear-sky GHI (Ineichen model) and a day/night flag.
2. **Night rules:** when the sun is below −2°, power and GHI must be 0.
3. **Capacity check:** power above 1.05 × capacity is impossible, so it is set to NaN.
4. **Flags, kept as columns and not deleted:** GHI above 1.1 × clear-sky (cloud enhancement) and daytime flatlines of 3 h or more (frozen sensor or curtailment).
5. **Gaps:** interpolate in time only for gaps **under 3 h**. Longer gaps stay NaN because inventing days of data would be dishonest.
6. **Target:** `capacity_factor = power / capacity`, so that sites of different sizes can be compared.

In [7]:
def flatline_flag(s, is_day, hours):
    same = s.diff().eq(0) & s.gt(0) & is_day.eq(1)
    run = same.groupby((~same).cumsum()).transform("sum")
    return (same & (run >= hours - 1)).astype(int)


def finalize(site, d, offset_min):
    cfg = SITES[site]
    d = add_solar_features(d, cfg["lat"], cfg["lon"], cfg["altitude"], "ghi", offset_min)
    cap = cfg["capacity_kw"] or float(d["power_kw"].quantile(0.999))

    night = d["solar_elevation"] < -2
    log(site, "night power > 1% capacity -> 0", (night & (d["power_kw"] > 0.01 * cap)).sum())
    log(site, "night GHI > 5 W/m2 -> 0", (night & (d["ghi"] > 5)).sum())
    d.loc[night & d["power_kw"].notna(), "power_kw"] = 0.0
    d.loc[night & d["ghi"].notna(), "ghi"] = 0.0

    over = d["power_kw"] > 1.05 * cap
    log(site, "power > 1.05 x capacity -> NaN", over.sum())
    d.loc[over, "power_kw"] = np.nan

    d["flag_ghi_above_clearsky"] = (d["ghi"] > 1.1 * d["ghi_clearsky"] + 50).astype(int)
    d["flag_flatline"] = flatline_flag(d["power_kw"], d["is_day"], FLATLINE_H)
    log(site, "flag: GHI > 1.1 x clear-sky (kept)", d["flag_ghi_above_clearsky"].sum())
    log(site, f"flag: daytime flatline >= {FLATLINE_H} h (kept)", d["flag_flatline"].sum())

    measured = [c for c in d.columns if c not in ("is_day",)
                and not c.startswith(("flag_", "solar_", "ghi_clearsky", "clearsky"))]
    for c in measured:
        before = d[c].isna().sum()
        d[c] = d[c].interpolate(method="time", limit=SHORT_GAP_H - 1, limit_area="inside")
        log(site, f"{c}: short gaps interpolated", before - d[c].isna().sum())

    d["clearsky_index"] = (d["ghi"] / d["ghi_clearsky"].where(d["ghi_clearsky"] > 20)).clip(0, 1.5)
    d["capacity_factor"] = d["power_kw"] / cap
    log(site, "capacity used (kW)", round(cap, 2))
    return d

In [8]:
primary = finalize("primary", hourly, offset_min=30)   # hourly means labelled at hour start -> geometry at :30
primary.head()

,power_kw,temp_c,humidity_pct,ghi,dhi,wind_dir,gti,dti,rain_mm,solar_zenith,solar_elevation,solar_azimuth,ghi_clearsky,is_day,clearsky_index,flag_ghi_above_clearsky,flag_flatline,capacity_factor
2023-12-31 14:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,132.280783,-42.280783,191.656906,0.0,0,NaN,0,0,NaN
2023-12-31 15:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,132.837051,-42.837051,172.915087,0.0,0,NaN,0,0,NaN
2023-12-31 16:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129.050660,-39.050660,155.322912,0.0,0,NaN,0,0,NaN
2023-12-31 17:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,121.764988,-31.764988,140.922981,0.0,0,NaN,0,0,NaN
2023-12-31 18:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,112.101844,-22.101844,129.863115,0.0,0,NaN,0,0,NaN


## Part B: Site A
Site A is already hourly and in UTC. The main steps are to drop the sparse inverter telemetry and to set power to NaN during inverter faults, because a fault is not weather and would mislead the model.

In [9]:
raw_a = load_site_a_raw()
a = raw_a[["active_power_kw", "inverter_status", "temp_c", "humidity_pct", "cloud_cover_pct",
           "wind_speed_kmh", "ghi_wm2", "dni_wm2", "dhi_wm2", "precip_mm"]].rename(columns={
    "active_power_kw": "power_kw", "ghi_wm2": "ghi", "dni_wm2": "dni",
    "dhi_wm2": "dhi", "precip_mm": "rain_mm"})

full = pd.date_range(a.index.min(), a.index.max(), freq="h", tz="UTC")
log("site_a", "missing hourly rows inserted", len(full) - len(a))
a = a.reindex(full)

fault = a["inverter_status"].eq("OFF: unexpected shutdown")
log("site_a", "power -> NaN during inverter fault", fault.sum())
a.loc[fault, "power_kw"] = np.nan
a = a.drop(columns="inverter_status")

site_a = finalize("site_a", a, offset_min=0)
site_a.head()

,power_kw,temp_c,humidity_pct,cloud_cover_pct,wind_speed_kmh,ghi,dni,dhi,rain_mm,solar_zenith,solar_elevation,solar_azimuth,ghi_clearsky,is_day,clearsky_index,flag_ghi_above_clearsky,flag_flatline,capacity_factor
2025-01-03 22:00:00+00:00,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,157.704057,-67.704057,357.083061,0.0,0,NaN,0,0,0.0
2025-01-03 23:00:00+00:00,0.0,0.1,68.0,73.0,8.7,0.0,0.0,0.0,0.0,155.030966,-65.030966,31.402694,0.0,0,NaN,0,0,0.0
2025-01-04 00:00:00+00:00,0.0,0.6,63.0,46.0,9.1,0.0,0.0,0.0,0.0,147.639450,-57.639450,56.096621,0.0,0,NaN,0,0,0.0
2025-01-04 01:00:00+00:00,0.0,-0.1,63.0,23.0,5.5,0.0,0.0,0.0,0.0,138.066698,-48.066698,72.775794,0.0,0,NaN,0,0,0.0
2025-01-04 02:00:00+00:00,0.0,-0.8,67.0,53.0,5.2,0.0,0.0,0.0,0.0,127.667089,-37.667089,85.280602,0.0,0,NaN,0,0,0.0


## Cleaning log

In [10]:
log_df = pd.DataFrame(LOG, columns=["site", "rule", "rows"])
log_df

,site,rule,rows
0,primary,duplicate timestamps dropped,0
1,primary,"humidity_pct outside [0, 100] clipped",3783
2,primary,"ghi outside [0, 1400] clipped",7
3,primary,"dhi outside [0, 800] clipped",0
4,primary,"gti outside [0, 1400] clipped",0
5,primary,"dti outside [0, 800] clipped",0
6,primary,"temp_c outside [-10, 50] clipped",0
7,primary,"wind_dir outside [0, 360] -> NaN",1014
8,primary,hourly power_kw: <80% coverage -> NaN,5992
9,primary,hourly temp_c: <80% coverage -> NaN,5474


## Validation checks
### 1. Is the timezone/location right?
If it is, measured GHI should line up best with the modelled clear-sky GHI at **lag 0**.

In [11]:
for name, d in [("primary", primary), ("site_a", site_a)]:
    lags = {lag: round(d["ghi"].corr(d["ghi_clearsky"].shift(lag)), 3) for lag in (-2, -1, 0, 1, 2)}
    print(name, lags)

primary {-2: np.float64(0.741), -1: np.float64(0.882), 0: np.float64(0.936), 1: np.float64(0.889), 2: np.float64(0.753)}
site_a {-2: np.float64(0.794), -1: np.float64(0.903), 0: np.float64(0.938), 1: np.float64(0.89), 2: np.float64(0.769)}


### 2. Sanity assertions

In [12]:
for name, d in [("primary", primary), ("site_a", site_a)]:
    step = d.index.to_series().diff().dropna()
    assert not d.index.duplicated().any(), "duplicate timestamps"
    assert (step == pd.Timedelta("1h")).all(), "irregular grid"
    assert d["capacity_factor"].dropna().between(0, 1.05).all(), "CF out of range"
    day_missing = d.loc[d["is_day"] == 1, "capacity_factor"].isna().mean() * 100
    print(f"{name}: {d.shape}, corr(power, GHI) = {d['power_kw'].corr(d['ghi']):.3f}, "
          f"night mean power = {d.loc[d.solar_elevation < -2, 'power_kw'].mean():.4f}, "
          f"daytime target missing = {day_missing:.1f}%")
print("All checks passed")

primary: (18746, 18), corr(power, GHI) = 0.996, night mean power = 0.0682, daytime target missing = 15.8%
site_a: (13031, 18), corr(power, GHI) = 0.924, night mean power = 0.0000, daytime target missing = 2.0%
All checks passed


### 3. Leakage check on the forecasting features (`src/features.py`)
For a forecast made at time *t* for *t+h*, features may only use data up to *t*. The check **modifies all future values** and confirms that none of the past features change. It runs for horizons of 1, 6, 24 and 48 h.

In [13]:
from features import build_features, chronological_split, daytime_trainable, leakage_check

for name, d in [("primary", primary), ("site_a", site_a)]:
    d.attrs["tz"] = SITES[name]["tz"]
    for h in (1, 6, 24, 48):
        leakage_check(d, h)
    X = build_features(d, horizon=1)
    train, test = chronological_split(daytime_trainable(X))
    print(f"{name}: leakage OK | {X.shape[1] - 1} features | "
          f"train {len(train)} rows ({train.index.min().date()} -> {train.index.max().date()}) | "
          f"test {len(test)} rows ({test.index.min().date()} -> {test.index.max().date()})")

primary: leakage OK | 41 features | train 6399 rows (2024-01-01 -> 2025-10-23) | test 1600 rows (2025-10-23 -> 2026-02-19)


site_a: leakage OK | 41 features | train 5224 rows (2025-01-04 -> 2026-04-01) | test 1307 rows (2026-04-01 -> 2026-06-30)


## Save as CSV (opens in Excel)

In [14]:
primary.to_csv(PROCESSED / "primary_hourly_clean.csv")
site_a.to_csv(PROCESSED / "site_a_hourly_clean.csv")
log_df.to_csv(PROCESSED / "cleaning_log.csv", index=False)
print([p.name for p in PROCESSED.iterdir()])

['cleaning_log.csv', 'primary_hourly_clean.csv', 'site_a_hourly_clean.csv']
